Name: Shaun Clarke

Course: CSC6314 Deep Learning

Instructor: Margaret Mulhall

Assignment:
You are hired by a retail company. Develop and train a Recurrent Neural Network (RNN)
that processes sequential text data to classify customer sentiment based on customer feedback on a product.
Your system will learn from a provided database of comments and provide a real-time interface for sentiment analysis,
helping customer service staff identify the emotional tone of feedback.


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import re
import os
import sys
from typing import Dict, List, Tuple

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Following best practice and defining my variables here
# Instead of searching throughout the file to change variables.
# Must be in my CWD
DATA_FILE: str = "./customer_data.txt"
# Every sequence must be exactly 10 tokens
SEQ_LENGTH: int = 10
# Dimensionality of each word's learned vector representation
EMBED_DIM: int = 16
# Size of LSTM hidden state, the memory vector
HIDDEN_SIZE: int = 32
# Model will have 2 LSTM layers
NUM_LAYERS: int = 2
# Mode will train for 200 epochs
NUM_EPOCHS: int = 400
# The step size for Adam optimizer
LEARNING_RATE: float = 0.001
PAD_IDX: int = 0

In [ ]:
# ==========================================
# 1. DATA LOADING & TOKENIZATION
# ==========================================

# The helper function raw customer comment texts
def clean_text(text: str) -> str:
    """
    This function makes all words lowercase and
    Uses regex to strip apostrophies and punctuation from input text.
    :param text: Raw string from the customer_data.txt file
    :return: Cleaned lower case string with apostrophe and punctuation stripped
    """

    # Making text lowercase
    text: str = text.lower()
    # remove apostrophie
    text.replace("'", "")
    # Using regex to remove remainin gpunctuations
    text: str = re.sub(r"[^a-z\s]", "", text)

    return text.strip()


# This method loads and parses the customer_data.txt file into (comment, label) pairs
def prepare_data(filepath: str) -> Tuple[List[str], List[int]]:
  """
  This method loads and parses the customer_data.txt into (comment, label) pairs
  :param filepath: Path to customer_data.txt
  :return: Tuple of (comments, labels)
  """
  # Lists to hold comments and labels
  comments = []
  labels = []

  # Checking if the file exists before openeing it.
  if not os.path.exists(filepath):
    print(f"ERROR: {filepath} not found.")
    # Exiting the program
    sys.exit()

  # Opening the file
  with open(filepath, "r") as f:
    # Iterating over each line
    for line in f:
      # splitting each line so we can separate comments from labels.
      split_line: str = line.split()
      # Adding labels to the list by slicing them
      labels. append(split_line[-1])
      # Using the clean function to slice and recreate the comments
      comments.append(clean_text(" ".join(split_line[:-1])))

  return comments, labels


# This function assigns indices to words
def build_vocab(comments: List[str]) -> Dict[str, int]:
  """
  This functions assigns indices to words and returns a dictionary of (word, index) pairs
  It starts the indices from 1 to reserve 0 for PAD_IDX
  :param comments: List of comments
  :return: Dictionary of (word, index) pairs
  """
  # Dictionary to hold vocabulary
  vocab: Dict[str, int] = {"PAD": PAD_IDX}

  # Getting vocabulary from comments
  for comment in comments:
    # splitting comments into individual words
    for word in comment.split():
      # Making sure word doesn't already exist
      if word not in vocab:
        vocab[word] = len(vocab)

  return vocab


# This function converts text comments to fixed length(10) integer tensors
def encode_and_pad(comments: List[str], vocab: Dict[str, int], seq_length: int) -> torch.Tensor:
  """
  This function converts text comments to fixed length(10) integer tensors.
  The LSTM requires all sequences in a batch to be thesame length.
  Hence why we pad short sequences with 0 and truncate the long ones.
  :param comments: List of comments
  :param vocab: Dictionary or vocab and indices
  :param seq_length: Length of each sequence
  :return: Tensor of encoded and padded comments, shape (num_samples, seq_length) dtype=torch.long
  """

  # List to hold encoded words
  encoded: List[List[int]] = []

  # Getting words from comments so we can get their indices
  for comment in comments:
    # splitting comments into individual words
    words: List = comment.split()
    # Using a list comprehension to convert each word to its index
    indices: List = [vocab.get(word, 0) for word in words]

    # Truncating to the last seq_length token if indices it is too long
    if len(indices) > seq_length:
      indices: List = indices[-seq_length:]

    # Padding with 0s if too short
    # basically, we subtract the lenght of the indices from th sequence length
    # Then multiply the ouput of the subtraction by [0] which gives [0,0]
    # then we add [0,0] to the indices
    if len(indices) < seq_length:
      indices: List = [0] * (seq_length - len(indices)) + indices

    # Adding encoded indices to list
    encoded.append(indices)

  return torch.tensor(encoded, dtype=torch.long)



In [ ]:
# ==========================================
# 2. RNN MODEL (LSTM IMPLEMENTATION)
# ==========================================

# This class is the LSTM bsed binry sentiment classifier
class SentimentLSTM(nn.Module):
  """
  This class is the LSTM based binary sentiment classifier.

  Architecture: Embedding > LSTM(2 layers) > Fully Connected

  Mental note for my understanding:
  - Embedding convert integer IDs into dense float vectors
  - LSTM processes th sequence step by step, while maiuntaining the hidden and cell states.
  - Linear maps the context vector to a single scalar value.
  - Sigmoid squashes the scalar value between 0 and 1.
  """

  def __init__(self, vocab_size: int, embed_dim: int, hidden_size: int, num_layers: int):
    super(SentimentLSTM, self).__init__()
    # Embedding layer that turns each integer word ID int a learnable float vector
    self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
    # LSTM layer, setting batch first to true rearranging the input shape so that batch is first
    self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers=num_layers, batch_first=True)
    # Fully connected layer, maps the context vector aka the hidden_size to a single sentiment score
    self.fc = nn.Linear(hidden_size, 1)
    # Sigmoid layer to squash the output to a 0 and 1 probability
    self.sigmoid = nn.Sigmoid()

  # The forward pass method that puts all the layers topgether
  def forward(self, x: torch.Tensor) -> torch.Tensor:

    # Converting each integer word ID int a learnable float vector
    x = self.embedding(x)
    # Passing embedding sequence through the LSTM layer one word at a time
    # output: every hidden state at every time step with a shape of (batch, seq_length, hidden_size)
    # h_n: final hidden state aka working memory per layer with a shape of (num_layers, batch, hidden_size)
    # c_n: the final cell state aka the long term memory per layer with a shape of (num_layers, batch, hidden_size)
    output, (h_n, c_n) = self.lstm(x)
    # h_n has one row per LSTM layer. We want h_n[-1] because that is the final hidden state of the deepest layer
    # after reading a full senstemnce. Which is basically the context vector, a single vector summarizing a full sentence.
    context = h_n[-1]
    # Linear layer that maps the context vector to 1 number
    x = self.fc(context)
    # Squashes the raw score into a value between 0 and 1
    x = self.sigmoid(x)

    return x



In [ ]:
# ==========================================
# 3. TRAINING & LIVE INFERENCE ENGINE
# ==========================================

# This function trains the LSTM model using BCELoss and Adam optimizer
def train(model: SentimentLSTM, X: torch.Tensor, Y: torch.Tensor, num_epochs: int, lr: float) -> None:
  """
  This function trains the LSTM model using BCELoss and Adam optimizer
  :param model: LSTM model
  :param X: Tensor of encoded
  :param Y: Tensor of labels
  :param num_epochs: Number of epochs to train for
  :param lr: Learning rate
  :return: None

  Mapping the training pattern for my mental note:
  1: optimizer.zer_grad() clears gradients from last step
  2: output = model(X) is the forward pass
  3: loss = criterion(output, Y) calculates the loss
  4: loss.backward() calculates the gradients
  5: optimizer.step() updates the weights
  """

  # Initializing criterion
  criterion = nn.BCELoss()
  # Initializing optimizer
  optimizer = optim.Adam(model.parameters(), lr=lr)

  # Setting the model to training mode
  model.train()

  for epoch in range(num_epochs):
    # Clearing gradients from last step
    optimizer.zero_grad()
    # Forward pass
    output = model(X)
    # Calculating loss
    loss = criterion(output, Y.float().unsqueeze(1))
    # Backward pass
    loss.backward()
    # Updating weights
    optimizer.step()

    if (epoch + 1) % 20 == 0:
      print(f"Epoch: {epoch + 1}, Loss: {loss.item():.4f}")


# This function classifies a user typed comment.
def predict(model: SentimentLSTM, vocab: Dict[str, int], text: str, seq_length: int) -> Tuple[str, float]:
  """
  This function classifies a user typed comment.
  :param model: The trained SentimentLSTM model
  :param vocab: The dictionary vocabulary built from training data
  :param text: User typed comment
  :param seq_length: Length of each sequence. This must match teh training sequence length
  :return: Tuple of (label, confidence)

  # TODO: Clean the input text using clean_text()
  # TODO: Encode and pad using the training vocab — same as training
  # TODO: Add batch dimension: tensor.unsqueeze(0) → shape (1, seq_length)
  # TODO: Run through model, get probability
  # TODO: confidence = prob.item() * 100
  # TODO: label = "Happy" if prob > 0.5 else "Angry"
  """

  # Switching model to eval mode to disable to disable training specific features that are not needed
  model.eval()

  # using torch.no_grad() to skip gradient computation because we do not need this for predictions.
  with torch.no_grad():
    # Cleaning the input text
    text = clean_text(text)
    # Encoding and padding the text
    encoded = encode_and_pad([text], vocab, seq_length)
    # Adding batch dimension - REMOVING THIS LINE
    # encoded = encoded.unsqueeze(0)
    # Running the data through the model
    output = model(encoded)
    # Getting the probability
    prob = output.squeeze()
    # Calculating confidence
    confidence = prob.item() * 100
    # Getting the label
    label = "Happy" if prob > 0.5 else "Angry"

    return label, confidence


In [ ]:
# ==========================================
# Main
# ==========================================

# Main is where the full pipeline is orchstrated
def main() -> None:
  """
  This is where the full pipeline is orchestrated.
  :return: None
  """

  # calling prepare data with the customer file
  comments, labels = prepare_data(DATA_FILE)
  # Calling build vocab with comments
  vocab = build_vocab(comments)
  # Calling encode and pad with comments, vocab, and seq length
  X_encoded = encode_and_pad(comments, vocab, SEQ_LENGTH)
  # Instantiating the model
  model = SentimentLSTM(len(vocab), EMBED_DIM, HIDDEN_SIZE, NUM_LAYERS)
  # Converting labels to tensors.
  # Ran into a ValueError: too many dimensions 'str' because the labels list are strings and not ints
  # So converting the 1,0 from strings to ints so i can create the tensor
  Y_labels = torch.tensor([int(label) for label in labels], dtype=torch.long)
  # Training the model
  train(model, X_encoded, Y_labels, NUM_EPOCHS, LEARNING_RATE)

  # The live inference loop
  print("Sentiment Classifier Ready. Type 'quit' to exit.")
  while True:
      user_input: str = input("Enter customer comment: ")
      if user_input.lower() == "quit":
          break
      label, confidence = predict(model, vocab, user_input, SEQ_LENGTH)
      print(f"Sentiment: {label}, Confidence: {confidence:.2f}%")


# Running the program
if __name__ == "__main__":
  main()


Epoch: 20, Loss: 0.6785
Epoch: 40, Loss: 0.6408
Epoch: 60, Loss: 0.4760
Epoch: 80, Loss: 0.2680
Epoch: 100, Loss: 0.1471
Epoch: 120, Loss: 0.0733
Epoch: 140, Loss: 0.0287
Epoch: 160, Loss: 0.0156
Epoch: 180, Loss: 0.0105
Epoch: 200, Loss: 0.0079
Epoch: 220, Loss: 0.0063
Epoch: 240, Loss: 0.0052
Epoch: 260, Loss: 0.0044
Epoch: 280, Loss: 0.0038
Epoch: 300, Loss: 0.0033
Epoch: 320, Loss: 0.0029
Epoch: 340, Loss: 0.0026
Epoch: 360, Loss: 0.0024
Epoch: 380, Loss: 0.0021
Epoch: 400, Loss: 0.0020
Sentiment Classifier Ready. Type 'quit' to exit.
Enter customer comment: i dont like it
Sentiment: Happy, Confidence: 99.92%
Enter customer comment: it solved my problem
Sentiment: Happy, Confidence: 99.80%
